### load models and basic infrence

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import pipeline
import torch

model_name = "Qwen/Qwen2.5-0.5B"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Save model and tokenizer locally
model_save_path = "./model_cache/qwen2.5-0.5b"
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)


('./model_cache/qwen2.5-0.5b\\tokenizer_config.json',
 './model_cache/qwen2.5-0.5b\\special_tokens_map.json',
 './model_cache/qwen2.5-0.5b\\vocab.json',
 './model_cache/qwen2.5-0.5b\\merges.txt',
 './model_cache/qwen2.5-0.5b\\added_tokens.json',
 './model_cache/qwen2.5-0.5b\\tokenizer.json')

### use the model to generate text


In [11]:
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

text = pipe("hello, how are you?", max_new_tokens=25)
# text = pipe("hello, who is trump?", max_new_tokens=25)
print(type(text))
print(text[0]['generated_text'])



Device set to use cpu


<class 'list'>
hello, how are you? I'm doing a project for school and I need help with this problem. I have a function that takes in a string and


### tokenize the text


In [33]:
input_prompt = ["how are you?",
                "who is trump?",
                "what is the weather in beijing?"
                ]

input_ids = tokenizer.encode(input_prompt[0], return_tensors="pt", padding=True).to(model.device)
print(input_ids.shape)

input_ids_batch = tokenizer.batch_encode_plus(input_prompt, return_tensors="pt", padding=True).to(model.device)
print(input_ids_batch.keys())
print(input_ids_batch['input_ids'].shape)
print(input_ids_batch['input_ids'])
print(input_ids_batch['input_ids'][2])
print(tokenizer.batch_decode(input_ids_batch['input_ids'][2]))
print(input_ids_batch['attention_mask'][2])



torch.Size([1, 4])
dict_keys(['input_ids', 'attention_mask'])
torch.Size([3, 8])
tensor([[  5158,    525,    498,     30, 151643, 151643, 151643, 151643],
        [ 14623,    374,  38055,     30, 151643, 151643, 151643, 151643],
        [ 12555,    374,    279,   9104,    304,    387,  23649,     30]])
tensor([12555,   374,   279,  9104,   304,   387, 23649,    30])
['what', ' is', ' the', ' weather', ' in', ' be', 'ijing', '?']
tensor([1, 1, 1, 1, 1, 1, 1, 1])


### chat template

In [66]:
import json
prompt_template = [
    {"role": "system", 
     "content": "You are a helpful assistant."
     },
    {"role": "user", 
     "content": "hello, how are you?"
    },
    {"role": "assistant", 
     "content": "I AM"
    }
]

tokennized_prompt = tokenizer.apply_chat_template(prompt_template,
                              tokenize=False,
                              add_generation_prompt=True,
                              padding=True,
                              return_tensors="pt",
                              )
print(f"tokennized_prompt: {tokennized_prompt}")

tokenized_ids = tokenizer.apply_chat_template(prompt_template,
          tokenize=True,
          padding=True,
          add_generation_prompt=True,
          return_tensors="pt",
          )
print(f"\ntokenized_ids: {tokenized_ids}")
decoded_prompt = tokenizer.batch_decode(tokenized_ids)
print(f"\ndecoded_prompt: {decoded_prompt[0]}")



tokennized_prompt: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
hello, how are you?<|im_end|>
<|im_start|>assistant
I AM<|im_end|>
<|im_start|>assistant


tokenized_ids: tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,  14990,     11,   1246,    525,
            498,     30, 151645,    198, 151644,  77091,    198,     40,   6769,
         151645,    198, 151644,  77091,    198]])

decoded_prompt: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
hello, how are you?<|im_end|>
<|im_start|>assistant
I AM<|im_end|>
<|im_start|>assistant



In [67]:
OUTPUT_IDS = model.generate(tokenized_ids, max_new_tokens=25)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(tokenized_ids, OUTPUT_IDS)
]
print(f"\nOUTPUT_IDS: {generated_ids}")
OUTPUT_TEXT = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
print(f"\nOUTPUT_TEXT: {OUTPUT_TEXT}")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



OUTPUT_IDS: [tensor([    40,   1079, 112169,   8948,    198,  34978,    198,  34978,    198,
         34978,    198,  34978,    198,  34978,    198,  34978,    198,  34978,
           198,  34978,    198,  34978,    198,  34978,    198])]

OUTPUT_TEXT: ['I am很高兴system\n Comey\n Comey\n Comey\n Comey\n Comey\n Comey\n Comey\n Comey\n Comey\n Comey\n']
